# Resume Parser Evaluation v3 — Human Review + Full Pipeline

**Dataset:** 16 fictional profiles × 14 job descriptions = 224 pairs. It includes adjacent-role partial matches and harder negative matches. The initial labels are *proposed labels for human review*, not truly human-validated ground truth. Review them before treating any metric as final.

**Methods compared**
1. Semantic similarity (Sentence Transformers)
2. TF-IDF lexical similarity
3. Hybrid semantic + TF-IDF
4. Full pipeline: semantic + TF-IDF + skill coverage

This notebook includes reviewer-label workflow, ranking metrics, method comparison, and CSV exports. It does not estimate hiring probability.

## 1. Install dependencies

In [1]:
%pip install -q sentence-transformers scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 2. Load and review labels

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_PATH = Path("human_review_resume_job_pairs_v3.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Place human_review_resume_job_pairs_v3.csv next to this notebook.")

data = pd.read_csv(DATA_PATH)
print(f"{len(data)} pairs | {data.resume_id.nunique()} profiles | {data.role.nunique()} job roles")
display(data[["resume_id","profile_focus","role","relevance","label_status","reviewer_label","reviewer_notes"]].head(20))

224 pairs | 16 profiles | 14 job roles


,resume_id,profile_focus,role,relevance,label_status,reviewer_label,reviewer_notes
0,R01,ML Engineer,ML Engineer,2,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN
1,R01,ML Engineer,Data Analyst,1,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN
2,R01,ML Engineer,Backend Python Developer,0,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN
3,R01,ML Engineer,Frontend Developer,0,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN
4,R01,ML Engineer,DevOps Engineer,0,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN
5,R01,ML Engineer,NLP Engineer,1,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN
6,R01,ML Engineer,Computer Vision Engineer,1,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN
7,R01,ML Engineer,Database Developer,0,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN
8,R01,ML Engineer,Cybersecurity Analyst,0,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN
9,R01,ML Engineer,Data Engineer,1,INITIAL_LABEL_REQUIRES_HUMAN_REVIEW,NaN,NaN


### Labeling protocol

For every pair, review the resume text and job description, then fill `reviewer_label` with:
- `0` = not relevant
- `1` = partially relevant / transferable skills
- `2` = strongly relevant

Use `reviewer_notes` to record the reason. The existing `relevance` column is only a proposed starter label. Save the reviewed CSV and set `USE_REVIEWER_LABELS = True` below. Ideally use two independent reviewers and resolve disagreements.

In [3]:
# Set True only after you have reviewed and saved reviewer_label values in the CSV.
USE_REVIEWER_LABELS = False

if USE_REVIEWER_LABELS:
    data["evaluation_label"] = pd.to_numeric(data["reviewer_label"], errors="coerce")
    if data["evaluation_label"].isna().any():
        raise ValueError("Some reviewer_label cells are blank or invalid. Complete all labels before evaluation.")
    if not data["evaluation_label"].isin([0,1,2]).all():
        raise ValueError("Reviewer labels must be 0, 1, or 2.")
else:
    print("WARNING: evaluating proposed labels, not human-validated labels.")
    data["evaluation_label"] = data["relevance"].astype(int)

print(data["evaluation_label"].value_counts().sort_index())

evaluation_label
0    173
1     37
2     14
Name: count, dtype: int64


## 3. Extract skill coverage

In [4]:
def normalize_skill(s):
    return str(s).strip().lower()

def skill_coverage(resume_text, required_skills):
    # Transparent baseline: normalized substring matching. This is not a production-grade skill extractor.
    resume = str(resume_text).lower()
    skills = [normalize_skill(x) for x in str(required_skills).split(";") if x.strip()]
    if not skills:
        return 0.0
    found = sum(1 for skill in skills if skill in resume)
    return found / len(skills)

data["skill_coverage"] = data.apply(
    lambda r: skill_coverage(r["resume_text"], r["required_skills"]), axis=1
)
data[["resume_id","role","skill_coverage","evaluation_label"]].head()

,resume_id,role,skill_coverage,evaluation_label
0,R01,ML Engineer,0.833333,2
1,R01,Data Analyst,0.285714,1
2,R01,Backend Python Developer,0.222222,0
3,R01,Frontend Developer,0.000000,0
4,R01,DevOps Engineer,0.000000,0


## 4. Compute similarity and full-pipeline scores

In [5]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

results=[]
for resume_id, group in data.groupby("resume_id", sort=True):
    resume_text = group["resume_text"].iloc[0]
    jobs = group.drop_duplicates("role").copy()
    jd_texts = jobs["job_description"].tolist()

    emb = model.encode([resume_text] + jd_texts, convert_to_numpy=True,
                       normalize_embeddings=True, show_progress_bar=False)
    semantic = emb[1:] @ emb[0]

    tfidf_matrix = TfidfVectorizer(stop_words="english", ngram_range=(1,2),
                                   sublinear_tf=True).fit_transform([resume_text] + jd_texts)
    lexical = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).ravel()

    # Min-max normalize lexical and semantic within this resume's candidate set.
    def minmax(x):
        x=np.asarray(x,dtype=float)
        span=x.max()-x.min()
        return (x-x.min())/span if span > 1e-12 else np.zeros_like(x)
    semantic_norm=minmax(semantic)
    lexical_norm=minmax(lexical)
    skills=jobs["skill_coverage"].to_numpy(dtype=float)

    hybrid = 0.7*semantic_norm + 0.3*lexical_norm
    full_pipeline = 0.45*semantic_norm + 0.25*lexical_norm + 0.30*skills

    for i, (_, r) in enumerate(jobs.iterrows()):
        results.append({
            "resume_id":resume_id, "profile_focus":r["profile_focus"], "role":r["role"],
            "relevance":int(r["evaluation_label"]),
            "semantic_score":float(semantic_norm[i]), "tfidf_score":float(lexical_norm[i]),
            "hybrid_score":float(hybrid[i]), "skill_coverage":float(skills[i]),
            "full_pipeline_score":float(full_pipeline[i])
        })
rankings=pd.DataFrame(results)
display(rankings.head())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,resume_id,profile_focus,role,relevance,semantic_score,tfidf_score,hybrid_score,skill_coverage,full_pipeline_score
0,R01,ML Engineer,ML Engineer,2,1.000000,1.000000,1.000000,0.833333,0.950000
1,R01,ML Engineer,Data Analyst,1,0.715337,0.111285,0.534121,0.285714,0.435437
2,R01,ML Engineer,Backend Python Developer,0,0.463682,0.115184,0.359133,0.222222,0.304120
3,R01,ML Engineer,Frontend Developer,0,0.000000,0.000000,0.000000,0.000000,0.000000
4,R01,ML Engineer,DevOps Engineer,0,0.337599,0.000000,0.236319,0.000000,0.151920


## 5. Ranking metrics

In [6]:
def precision_at_k(rels,k):
    rels=np.asarray(rels)
    return float(np.sum(rels[:k]>0)/k)
def recall_at_k(rels,k):
    rels=np.asarray(rels)
    total=np.sum(rels>0)
    return float(np.sum(rels[:k]>0)/total) if total else np.nan
def dcg_at_k(rels,k):
    rels=np.asarray(rels[:k],dtype=float)
    return float(np.sum((2**rels-1)/np.log2(np.arange(2,len(rels)+2)))) if len(rels) else 0.0
def ndcg_at_k(rels,k):
    ideal=dcg_at_k(sorted(rels,reverse=True),k)
    return dcg_at_k(rels,k)/ideal if ideal else np.nan
def mrr(rels):
    for rank, rel in enumerate(rels,start=1):
        if rel>0: return 1/rank
    return 0.0

SCORE_COLS=["semantic_score","tfidf_score","hybrid_score","full_pipeline_score"]
metric_rows=[]
for method in SCORE_COLS:
    for rid, group in rankings.groupby("resume_id"):
        ordered=group.sort_values(method,ascending=False)
        rels=ordered["relevance"].tolist()
        metric_rows.append({
            "resume_id":rid,"method":method,
            "Precision@1":precision_at_k(rels,1),
            "Recall@3":recall_at_k(rels,3),
            "nDCG@3":ndcg_at_k(rels,3),
            "MRR":mrr(rels),
            "top_role":ordered.iloc[0]["role"],
            "top_relevance":int(ordered.iloc[0]["relevance"])
        })
per_resume_metrics=pd.DataFrame(metric_rows)
metric_cols=["Precision@1","Recall@3","nDCG@3","MRR"]
comparison=per_resume_metrics.groupby("method")[metric_cols].mean().sort_values("nDCG@3",ascending=False)
display(comparison)

,Precision@1,Recall@3,nDCG@3,MRR
method,,,,
full_pipeline_score,1.0,0.766667,0.931782,1.0
hybrid_score,1.0,0.735417,0.923175,1.0
semantic_score,1.0,0.735417,0.923175,1.0
tfidf_score,1.0,0.683333,0.893992,1.0


## 6. Inspect top-ranked jobs and method disagreements

In [7]:
display(per_resume_metrics.sort_values(["resume_id","method"]))
top_roles=per_resume_metrics.pivot(index="resume_id",columns="method",values="top_role")
display(top_roles)

,resume_id,method,Precision@1,Recall@3,nDCG@3,MRR,top_role,top_relevance
48,R01,full_pipeline_score,1.0,0.60,1.0,1.0,ML Engineer,2
32,R01,hybrid_score,1.0,0.60,1.0,1.0,ML Engineer,2
0,R01,semantic_score,1.0,0.60,1.0,1.0,ML Engineer,2
16,R01,tfidf_score,1.0,0.60,1.0,1.0,ML Engineer,2
49,R02,full_pipeline_score,1.0,0.75,1.0,1.0,Data Analyst,2
...,...,...,...,...,...,...,...,...
30,R15,tfidf_score,1.0,1.00,1.0,1.0,Android Developer,2
63,R16,full_pipeline_score,1.0,0.75,1.0,1.0,Data Analyst,1
47,R16,hybrid_score,1.0,0.75,1.0,1.0,Data Analyst,1
15,R16,semantic_score,1.0,0.75,1.0,1.0,Data Analyst,1


method,full_pipeline_score,hybrid_score,semantic_score,tfidf_score
resume_id,,,,
R01,ML Engineer,ML Engineer,ML Engineer,ML Engineer
R02,Data Analyst,Data Analyst,Data Analyst,Data Analyst
R03,Backend Python Developer,Backend Python Developer,Backend Python Developer,Backend Python Developer
R04,Frontend Developer,Frontend Developer,Frontend Developer,Frontend Developer
R05,DevOps Engineer,DevOps Engineer,DevOps Engineer,DevOps Engineer
R06,NLP Engineer,NLP Engineer,NLP Engineer,NLP Engineer
R07,Computer Vision Engineer,Computer Vision Engineer,Computer Vision Engineer,Computer Vision Engineer
R08,Database Developer,Database Developer,Database Developer,Database Developer
R09,Product Analyst,Product Analyst,Product Analyst,Product Analyst


## 7. Export all outputs

In [8]:
rankings.to_csv("resume_job_rankings_v3.csv",index=False)
per_resume_metrics.to_csv("resume_metrics_by_profile_v3.csv",index=False)
comparison.to_csv("resume_method_comparison_v3.csv")
print("Saved rankings, per-profile metrics, and aggregate method comparison.")

Saved rankings, per-profile metrics, and aggregate method comparison.


## 8. Before reporting results

- If `USE_REVIEWER_LABELS` is False, results use proposed synthetic labels and must be described as a pipeline check only.
- For credible evaluation, review labels manually (ideally two reviewers), keep a separate held-out set, and avoid tuning weights on the test set.
- The skill coverage function uses simple substring matching; synonyms and resume phrasing can be missed. Replace it with your project's curated skill normalization before deployment.
- The full-pipeline weights (45% semantic, 25% TF-IDF, 30% skill coverage) are initial experimental weights, not validated optimal weights.
- Do not interpret scores as hiring probabilities or use automated rankings as the sole basis for hiring decisions.